# Health Retailers — Cash Anomaly Detection (cahier §11.3) — v2, executed on the real files
## Complete, executable Data Preparation Lab

**Objective:** transform raw cash-register and treasury exports into a traceable, model-ready dataset to
**prioritise cash operations and cancellations for review**.

**Analytical grain:** **one row = one cash-register operation** (`CaisseOperations.Id`), plus a second, small table:
**one row = one cash discrepancy** (expected vs counted cash at opening/closing).

**What changed compared with the first version of this notebook:**
- the joins of v1 used `PaymentId → PaymentsList.Id` and `TransactionId → Transactions.Id`. Executed on the real
  files they match **0 of 9,707 operations**. The real key chain is
  `CaisseOperations.PaymentId → TresoreriePayments.Id → SalePaymentId → PaymentsList.Id` (100 % match);
- `DomainEvents.csv` is used: it is the only source of register names and cash discrepancies;
- hours are converted from UTC to Paris time before building time features;
- both sub-cases of the cahier are covered: **(A) unsupervised anomaly scoring** and **(B) classification of
  cancellations** (`is_cancelled` is the only label available).

**Team division:** Person 1 = cash sources, cleaning, monetary/temporal features, discrepancies. Person 2 = treasury
and payment joins, user behaviour features, encoding, PCA and the two sub-cases. Both validate the final dataset.

# 1 — Business objective and definition of an anomaly

**Problem:** six registers record thousands of operations. The finance manager wants to know which operations, and
which cancellations, deserve a check. An alert is **a reason to look**, never proof of fraud.

**Scoring moment:** an operation is scored **when it is created**. Therefore its status, cancellation date,
cancelling user and time-to-cancel are **not inputs** (they are known only later). User benchmarks only count
cancellations that **had already happened** at the time of the operation.

**Full workflow**
1. **Data Understanding:** five sources profiled separately.
2. **Data Cleaning:** dates (UTC, month/day/year), amounts, Booleans, malformed event log, quality flags, quarantine.
3. **Data Transformation:** validated key chain, register and user names, discrepancy table.
4. **Feature Engineering:** amount, time (Paris), register and user history computed on the past only.
5. **Feature Selection:** relevance, redundancy, leakage audit.
6. **Dimensionality Reduction:** imputation, one-hot encoding, scaling, PCA.
7. **Optional extensions:** (A) Isolation Forest review queue, (B) cancellation classifier.
8. **Validation and reporting.**

## 0 — Setup and source files

| Source | Purpose | Key fields |
|---|---|---|
| `CaisseOperations.csv` | Main table: one cash operation | `Id`, `CaisseId`, `PaymentId`, `AddedAtUtc`, `AddedByUserId`, `Amount`, `Status`, `CancelledAtUtc` |
| `TresoreriePayments.csv` | Treasury record of each operation | `Id` (= `CaisseOperations.PaymentId`), `SalePaymentId`, `IsPurchase`, `Removed` |
| `PaymentsList.csv` | Payment: customer, mode | `Id` (= `SalePaymentId`), `Company_Id`, `PaymentMode_Id` |
| `DomainEvents.csv` | Event log: register names, openings, closings, discrepancies | `AggregateId`, `EventName`, `Data` (JSON), `UtcTimestamp` |
| `users_csv.csv` | Cashier names | `Id`, `FullName` |

In [1]:
# STEP 0: Import the libraries and define every input/output path in one place.
# Raw exports are read from the project folders and are never modified.
# All generated evidence (profiles, audits, matrices, figures) goes to outputs/cash_anomaly/.
# Change PROJECT if the health-retailers repository is stored elsewhere on your computer.

# Run once if needed:
# %pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import io, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Locate the data branch checked out beside this branch, or set HEALTH_RETAILERS_DATA.
import os
_candidates = [Path(os.environ['HEALTH_RETAILERS_DATA']).expanduser()] if os.environ.get('HEALTH_RETAILERS_DATA') else []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend([_base / 'data', _base.parent / 'data'])
PROJECT = next((_p.resolve() for _p in _candidates if (_p / 'original').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Check out the data branch beside ml-notebooks, or set HEALTH_RETAILERS_DATA to its root.')
RAW_DIR = PROJECT / 'original'                       # branch data-original: the 29 raw CSV files
SYNTH_DIR = PROJECT / 'generated'   # branch data-external: generated months
EXT_DIR = PROJECT / 'external'                        # weather and epidemics downloads
CLEAN_DIR = PROJECT / 'cleaned'                      # branch data-cleaned (product catalogue key)
OUTPUT_DIR = Path('outputs') / 'cash_anomaly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REAL_START = pd.Timestamp('2024-09-30')
REAL_END = pd.Timestamp('2025-04-30 23:59:59')       # last day of the real export
SYNTH_END = pd.Timestamp('2026-09-29 23:59:59')      # last day of the generated data
INCLUDE_SYNTHETIC = False                  # True = add the generated months (tagged, never mixed silently)

for folder in [RAW_DIR, SYNTH_DIR, EXT_DIR, CLEAN_DIR]:
    print(f'{str(folder):60s}', 'found' if folder.exists() else 'MISSING')

Matplotlib is building the font cache; this may take a moment.


data\original found
data\generated found
data\external found
data\cleaned found


### Guided code explanation — Reading and conversion helpers

**Why this step exists:** the 29 raw files were not exported consistently. Some are UTF-8, others Windows-1252
or DOS code page 850; some use `;`, others `,`; amounts use a decimal comma; dates mix `MM/DD/YYYY` and
`DD/MM/YYYY`. One set of helpers guarantees that every file follows the same rules.

- `read_text()` tries UTF-8 first, then keeps the legacy code page that produces real French letters
  (`pièce` instead of `pi�ce`).
- `load_csv()` detects `;` or `,` from the header line and reads **every column as text** (`dtype='string'`),
  so identifiers and amounts are never converted by accident before we inspect them.
- `parse_amount()` turns `42,38` into `42.38`; unreadable values become `NaN` and are counted.
- `parse_bool()` maps `VRAI/FAUX`, `TRUE/FALSE`, `1/0` to one Boolean type.
- `parse_date()` uses an **explicit format** per column. We never let pandas guess between day-first and
  month-first, because `03/04/2025` is ambiguous.
- `profile()` returns the Data Understanding evidence: type, missing values, distinct values.

**What to tell your professor:** "Every conversion is explicit and audited: a value that cannot be parsed
becomes missing and is counted, it is never invented."

In [2]:
# Helpers shared by every step of the notebook.
# Every column is read as text first; conversions are explicit and audited later.

def read_text(path):
    raw = path.read_bytes()
    try:
        return raw.decode('utf-8-sig')
    except UnicodeDecodeError:
        # Legacy export: keep the code page that yields real French letters.
        candidates = [raw.decode(cp, errors='replace') for cp in ('cp1252', 'cp850')]
        score = lambda t: sum(t.count(c) for c in 'éèàçêù°') - 3 * sum(t.count(c) for c in 'Š‚ø')
        return max(candidates, key=score)

def load_csv(path):
    text = read_text(path)
    header = text.split('\n', 1)[0]
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(io.StringIO(text), sep=sep, dtype='string')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def parse_amount(series):
    s = series.astype('string').str.strip().str.replace('\u00a0', '', regex=False).str.replace(' ', '', regex=False)
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce').astype('float64')

def parse_bool(series):
    mapping = {'vrai': True, 'true': True, '1': True, 'faux': False, 'false': False, '0': False}
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')

def parse_date(series, fmt):
    return pd.to_datetime(series.astype('string').str.strip(), format=fmt, errors='coerce')

def profile(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_%': (100 * df.isna().mean()).round(2),
        'unique_values': df.nunique(dropna=True),
    })

def safe_left_merge(left, right, left_on, right_on, label):
    """Left join that refuses to multiply rows: the lookup key must be unique."""
    right = right.dropna(subset=[right_on])
    if right[right_on].duplicated().any():
        raise ValueError(f'{label}: lookup key {right_on} is not unique, aggregate it first')
    n = len(left)
    out = left.merge(right, how='left', left_on=left_on, right_on=right_on,
                     suffixes=('', '_' + label), validate='many_to_one')
    assert len(out) == n, f'{label} multiplied rows!'
    matched = out[right_on].notna().mean() * 100
    print(f'{label:28s} {n:>7,} rows kept | {matched:5.1f} % matched')
    return out

conversion_log = []
def convert(df, name, amounts=(), bools=(), dates=None):
    """Apply the conversions and record how many non-empty values could not be parsed."""
    df = df.copy()
    for col in amounts:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_amount(df[col])
            conversion_log.append([name, col, 'amount', int(before - df[col].notna().sum())])
    for col in bools:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_bool(df[col])
            conversion_log.append([name, col, 'boolean', int(before - df[col].notna().sum())])
    for col, fmt in (dates or {}).items():
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_date(df[col], fmt)
            conversion_log.append([name, col, f'date {fmt}', int(before - df[col].notna().sum())])
    return df

print('Helpers ready.')

Helpers ready.


## 1 — Data Understanding: profile every dataset separately

### Guided code explanation — Profiling, and the malformed event log

- `DomainEvents.csv` is **malformed**: every whole line is wrapped in quotes and inner quotes are doubled, so a normal
  CSV reader sees one single column. `read_events()` removes the outer quotes and un-doubles the inner ones before
  parsing. It is also encoded in DOS code page 850.
- The `Data` column holds JSON (`{"caisseId": ..., "name": "melanie"}`); it is parsed in Step 3.
- `users_csv.csv` contains password hashes: dropped immediately, never displayed.

**Presentation use:** the overview table and the profile of `CaisseOperations.csv`.

In [3]:
# STEP 1: Read each dataset independently.
def read_events(path):
    lines = read_text(path).splitlines()
    fixed = [lines[0]] + [l[1:-1].replace('""', '"') if l.startswith('"') and l.endswith('"') else l for l in lines[1:]]
    return pd.read_csv(io.StringIO('\n'.join(fixed)), dtype='string')

SOURCES = {'cash': 'CaisseOperations.csv', 'treasury': 'TresoreriePayments.csv', 'payments': 'PaymentsList.csv',
           'events': 'DomainEvents.csv', 'users': 'users_csv.csv'}
def read_source(folder, key):
    return read_events(folder / SOURCES[key]) if key == 'events' else load_csv(folder / SOURCES[key])

raw = {k: read_source(RAW_DIR, k) for k in SOURCES}
raw['users'] = raw['users'].drop(columns=['Password', 'PasswordSalt'], errors='ignore')
synth = {k: read_source(SYNTH_DIR, k) for k in ('cash', 'treasury', 'payments', 'events')} if INCLUDE_SYNTHETIC else {}

overview = pd.DataFrame([[SOURCES[k], len(df), df.shape[1], int(df.duplicated().sum()),
                          int((df.isna().mean() == 1).sum())] for k, df in raw.items()],
                        columns=['file', 'rows', 'columns', 'exact_duplicates', 'empty_columns'])
display(overview)
overview.to_csv(OUTPUT_DIR / 'sources_overview.csv', index=False)
print('\n=== CaisseOperations.csv ===')
display(raw['cash'].head(3))
p = profile(raw['cash']); display(p); p.to_csv(OUTPUT_DIR / 'profile_cash.csv', index_label='column')
display(raw['events']['EventName'].value_counts().to_frame('events'))

,file,rows,columns,exact_duplicates,empty_columns
0,CaisseOperations.csv,9707,12,0,2
1,TresoreriePayments.csv,10280,11,0,2
2,PaymentsList.csv,10280,10,0,0
3,DomainEvents.csv,21743,8,0,0
4,users_csv.csv,7,4,0,0



=== CaisseOperations.csv ===


,Id,CaisseId,TransactionId,AddedAtUtc,CancelledAtUtc,AddedByUserId,CancelledByUserId,Amount,Status,DepositId,WithdrawalId,PaymentId
0,EE1B100C-59D6-5B16-A092-000076A66B60,BD92D636-F45D-471D-87FF-4852A2880606,A81A1BDD-ECDF-5AA9-8ECA-D64144AF56C6,12/2/2024 14:11,<NA>,6,<NA>,81.263,0,<NA>,<NA>,A81A1BDD-ECDF-5AA9-8ECA-D64144AF56C6
1,C9679F6E-B4F6-531B-8A2B-0006E4888CB7,BD92D636-F45D-471D-87FF-4852A2880606,FE3AFDDD-0C84-5EF3-BC93-1A10395F0F5B,1/14/2025 16:01,<NA>,6,<NA>,49.692,0,<NA>,<NA>,FE3AFDDD-0C84-5EF3-BC93-1A10395F0F5B
2,AB3A7933-5CDB-5337-88A0-000A5B8EF1E2,BD92D636-F45D-471D-87FF-4852A2880606,BB36DA14-E124-5EFF-8FB2-E67917A4B56C,11/5/2024 11:49,<NA>,6,<NA>,45.616,0,<NA>,<NA>,BB36DA14-E124-5EFF-8FB2-E67917A4B56C


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.00,9707
CaisseId,string,0,0.00,6
TransactionId,string,0,0.00,9707
AddedAtUtc,string,0,0.00,9347
CancelledAtUtc,string,9605,98.95,93
AddedByUserId,string,0,0.00,6
CancelledByUserId,string,9605,98.95,4
Amount,string,0,0.00,6811
Status,string,0,0.00,3
DepositId,string,9707,100.00,0


,events
EventName,
Tresorerie.PaymentAdded,10280
Tresorerie.CaisseTransactionAdded,9707
Tresorerie.BankAccountTransactionAdded,521
Tresorerie.CaisseDiscrepancyIdentified,437
Tresorerie.CaisseClosed,264
Tresorerie.CaisseOpened,262
Tresorerie.PaymentRemoved,112
Tresorerie.CaisseTransactionCancelled,102
Tresorerie.BankAccountTransactionCancelled,12


## 2 — Data Cleaning: standardize without destroying anomalies

- `AddedAtUtc`, `CancelledAtUtc`, `CreatedAtUtc`, `UtcTimestamp`: **month/day/year**, UTC.
  `PaymentsList.PaymentDate`: **day/month/year**. Each format is explicit.
- Amounts with a decimal comma are converted; **negative and extreme amounts are kept** (refunds, withdrawals, or
  anomalies).
- `TresoreriePayments.Removed = TRUE` and `PaymentsList.IsRemoved = VRAI` mark records deleted in the software: they are
  flagged, and the operation keeps its row.
- `PaymentId` is checked against `TransactionId`: if they are identical on every row, one is redundant.

### Guided code explanation — Quality flags and quarantine

- `flag_missing_key` / `flag_missing_timestamp`: unusable rows → quarantine (kept in a CSV, not erased).
- `flag_zero_amount`, `flag_negative_amount`: business signals, **kept**.
- `flag_duplicate_operation_id`: must be 0, otherwise the one-row-per-operation table is impossible.
- `flag_cancel_before_create`: logical inconsistency in the dates.
- `is_cancelled` = a cancellation date exists. It is the **label** of sub-case B and is never an input.

**What to tell the professor:** "Structural errors are quarantined; unusual business values are kept because they are
what we are looking for."

In [4]:
# STEP 2: Conversions, flags, quarantine.
def stack(key):
    return pd.concat([raw[key]] + ([synth[key]] if key in synth else []), ignore_index=True)

cash = convert(stack('cash'), 'cash', amounts=['Amount'],
               dates={'AddedAtUtc': '%m/%d/%Y %H:%M', 'CancelledAtUtc': '%m/%d/%Y %H:%M'})
treasury = convert(stack('treasury'), 'treasury', amounts=['Amount'], bools=['IsPurchase', 'Removed'],
                   dates={'CreatedAtUtc': '%m/%d/%Y %H:%M'})
payments = convert(stack('payments'), 'payments', amounts=['Amount'], bools=['IsRemoved'],
                   dates={'PaymentDate': '%d/%m/%Y %H:%M'})
events = convert(stack('events'), 'events', dates={'UtcTimestamp': '%m/%d/%Y %H:%M'})
for name, df in (('cash', cash), ('treasury', treasury), ('payments', payments), ('events', events)):
    print(f'{name}: {len(df):,} rows, exact duplicates = {int(df.duplicated().sum())}')
cash, treasury, payments, events = (d.drop_duplicates() for d in (cash, treasury, payments, events))

print('PaymentId identical to TransactionId on every row:', bool((cash['PaymentId'] == cash['TransactionId']).all()))
cash['flag_missing_key'] = cash['Id'].isna() | cash['CaisseId'].isna()
cash['flag_missing_timestamp'] = cash['AddedAtUtc'].isna()
cash['flag_zero_amount'] = cash['Amount'].eq(0)
cash['flag_negative_amount'] = cash['Amount'].lt(0)
cash['flag_duplicate_operation_id'] = cash['Id'].duplicated(keep=False)
cash['flag_cancel_before_create'] = cash['CancelledAtUtc'] < cash['AddedAtUtc']
cash['is_cancelled'] = cash['CancelledAtUtc'].notna()
flags = [c for c in cash if c.startswith('flag_')]
display(cash[flags + ['is_cancelled']].sum().to_frame('operations'))
display(pd.crosstab(cash['Status'], cash['is_cancelled']).rename_axis('Status / is_cancelled'))
if cash['flag_duplicate_operation_id'].any():
    raise ValueError('Non-unique operation ids: reconcile before going further.')
quarantine = cash[cash['flag_missing_key'] | cash['flag_missing_timestamp']]
quarantine.to_csv(OUTPUT_DIR / 'cash_quarantine.csv', index=False)
base = cash.drop(quarantine.index).copy()
print(f'Retained operations: {len(base):,} | quarantined: {len(quarantine):,}')
conversion_audit = pd.DataFrame(conversion_log, columns=['dataset', 'column', 'conversion', 'unparseable_non_null'])
display(conversion_audit)
conversion_audit.to_csv(OUTPUT_DIR / 'conversion_audit.csv', index=False)

cash: 9,707 rows, exact duplicates = 0
treasury: 10,280 rows, exact duplicates = 0
payments: 10,280 rows, exact duplicates = 0
events: 21,743 rows, exact duplicates = 0


PaymentId identical to TransactionId on every row: True


,operations
flag_missing_key,0
flag_missing_timestamp,0
flag_zero_amount,0
flag_negative_amount,29
flag_duplicate_operation_id,0
flag_cancel_before_create,0
is_cancelled,102


is_cancelled,False,True
Status / is_cancelled,,
0,9576,3
1,29,0
2,0,99


Retained operations: 9,707 | quarantined: 0


,dataset,column,conversion,unparseable_non_null
0,cash,Amount,amount,0
1,cash,AddedAtUtc,date %m/%d/%Y %H:%M,0
2,cash,CancelledAtUtc,date %m/%d/%Y %H:%M,0
3,treasury,Amount,amount,0
4,treasury,IsPurchase,boolean,0
5,treasury,Removed,boolean,0
6,treasury,CreatedAtUtc,date %m/%d/%Y %H:%M,0
7,payments,Amount,amount,0
8,payments,IsRemoved,boolean,0
9,payments,PaymentDate,date %d/%m/%Y %H:%M,0


**Reading the crosstab:** `Status = 2` matches the cancelled operations almost exactly. `Status` is updated *after*
the cancellation, so using it as an input would hand the answer to the model: it is excluded in Step 5.

## 3 — Controlled Data Transformation and Joins

The key chain was **verified on the data** (match rate printed by `safe_left_merge`):

`CaisseOperations.PaymentId` → `TresoreriePayments.Id` → `SalePaymentId` → `PaymentsList.Id`

- Register names: `CaisseAdded` events of `DomainEvents` (`caisseId` → `name`).
- Cashier names: `AddedByUserId` → `users.Id`.
- Discrepancies: `CaisseDiscrepancyIdentified` events, one row each, with expected and counted cash.

### Guided code explanation — Safe joins

- `safe_left_merge()` refuses a lookup with duplicate keys and asserts the row count is unchanged.
- `PaymentMode_Id` is **empty for most payments in the source export** (only a few hundred are filled). It is kept as a
  category with an explicit `NON_RENSEIGNE` value instead of being guessed.
- `is_supplier_payment` (`IsPurchase`) separates money paid **to** suppliers from customer payments.
- Discrepancies: `detected_at = OPENING` when the event refers to an opening operation. An opening discrepancy with
  counted cash = 0 usually means the cash was **taken out the evening before** — flagged as `overnight_removal`, not as a
  loss.

In [5]:
# STEP 3: Enrich operations using verified keys only.
analytical = base.copy()
tre = treasury[['Id', 'SalePaymentId', 'IsPurchase', 'Removed']].rename(
    columns={'Id': 'treasury_id', 'IsPurchase': 'is_supplier_payment', 'Removed': 'treasury_removed'})
analytical = safe_left_merge(analytical, tre, 'PaymentId', 'treasury_id', 'operation -> treasury')

pay = payments[['Id', 'Company_Id', 'PaymentMode_Id', 'IsRemoved']].rename(
    columns={'Id': 'payment_id', 'Company_Id': 'company_id', 'PaymentMode_Id': 'payment_mode_id',
             'IsRemoved': 'payment_removed'})
analytical = safe_left_merge(analytical, pay, 'SalePaymentId', 'payment_id', 'treasury -> payment')
MODES = {'1': 'cheque', '5': 'bank_card', '7': 'credit', '11': 'cash', '21': 'tokens'}
analytical['payment_mode'] = analytical['payment_mode_id'].map(MODES).fillna('NON_RENSEIGNE')
analytical['is_walk_in_customer'] = analytical['company_id'].eq('2').fillna(False).astype('int8')
display(analytical['payment_mode'].value_counts().to_frame('operations'))

def event_json(name):
    e = events[events['EventName'].str.endswith(name, na=False)].copy()
    e['data'] = e['Data'].map(json.loads)
    return e

registers = event_json('CaisseAdded')
registers = pd.DataFrame({'register_key': registers['data'].map(lambda d: d['caisseId'].upper()),
                          'register': registers['data'].map(lambda d: 'Caisse ' + d.get('name', '?'))}).drop_duplicates('register_key')
analytical = safe_left_merge(analytical, registers, 'CaisseId', 'register_key', 'register name')
users = raw['users'][['Id', 'FullName']].rename(columns={'Id': 'user_key', 'FullName': 'cashier'})
analytical = safe_left_merge(analytical, users, 'AddedByUserId', 'user_key', 'cashier name')
assert analytical['Id'].is_unique and len(analytical) == len(base)

disc = event_json('CaisseDiscrepancyIdentified')
discrepancies = pd.DataFrame({
    'event_id': disc['Id'], 'utc_time': disc['UtcTimestamp'], 'user_id': disc['UserId'],
    'register_key': disc['AggregateId'].str.upper(),
    'expected': disc['data'].map(lambda d: d.get('expectedBalance')),
    'found': disc['data'].map(lambda d: d.get('foundBalance')),
    'difference': disc['data'].map(lambda d: d.get('amount')),
    'detected_at': disc['data'].map(lambda d: 'OPENING' if 'openCaisseOperationId' in d.get('source', {}) else 'CLOSING')})
discrepancies = discrepancies.merge(registers, on='register_key', how='left', validate='many_to_one')
for c in ['expected', 'found', 'difference']:
    discrepancies[c] = pd.to_numeric(discrepancies[c], errors='coerce').astype('float64')
discrepancies['overnight_removal'] = discrepancies['detected_at'].eq('OPENING') & discrepancies['found'].eq(0)
display(discrepancies.groupby(['detected_at', 'overnight_removal']).size().to_frame('discrepancies'))
display(discrepancies.head())

operation -> treasury          9,707 rows kept | 100.0 % matched
treasury -> payment            9,707 rows kept | 100.0 % matched


,operations
payment_mode,
NON_RENSEIGNE,9645
cash,62


register name                  9,707 rows kept | 100.0 % matched
cashier name                   9,707 rows kept | 100.0 % matched


discrepancies
detected_at overnight_removal               
CLOSING     False                        243
OPENING     False                          4
            True                         190

,event_id,utc_time,user_id,register_key,expected,found,difference,detected_at,register,overnight_removal
0,38,2024-10-12 12:58:00,6,BD92D636-F45D-471D-87FF-4852A2880606,587.650,588.0,0.350,CLOSING,Caisse melanie,False
1,41,2024-10-14 07:22:00,6,BD92D636-F45D-471D-87FF-4852A2880606,588.000,0.0,-588.000,OPENING,Caisse melanie,True
2,253,2024-10-16 17:54:00,6,BD92D636-F45D-471D-87FF-4852A2880606,3165.789,0.0,-3165.789,CLOSING,Caisse melanie,False
3,396,2024-10-18 11:15:00,6,BD92D636-F45D-471D-87FF-4852A2880606,1499.425,0.0,-1499.425,CLOSING,Caisse melanie,False
4,467,2024-10-18 17:58:00,6,BD92D636-F45D-471D-87FF-4852A2880606,2468.398,0.0,-2468.398,CLOSING,Caisse melanie,False


## 4 — Feature Engineering

| Signal | Features |
|---|---|
| Amount | `amount`, `amount_log_signed`, `gap_vs_register_median`, `amount_z_vs_user` |
| Time (Paris) | `hour`, `weekday`, `month`, `is_saturday`, `minutes_since_previous_on_register`, `ops_last_60min_register` |
| User behaviour | `user_ops_today_before`, `user_past_cancel_rate` |
| Context | `register`, `cashier`, `payment_mode`, `is_supplier_payment`, `is_walk_in_customer` |

### Leakage rule
`Status`, `CancelledAtUtc`, `CancelledByUserId` and the time-to-cancel are **known only after** the operation:
they are never inputs. `user_past_cancel_rate` counts only cancellations **already recorded** before the operation.

### Guided code explanation — Feature engineering

- `tz_localize('UTC').tz_convert('Europe/Paris')`: the store works in Paris time; a 07:30 UTC operation is 08:30 or
  09:30 locally depending on the season.
- `shift(1).expanding().median()` per register: the usual amount of the register **before** this operation.
- `amount_z_vs_user`: how far the amount is from what this cashier usually records (past only, ≥ 10 operations).
- `ops_last_60min_register`: number of earlier operations on the same register in the previous hour
  (`rolling('60min')` on a time index, minus the current operation).
- `user_past_cancel_rate`: for each operation, `np.searchsorted` counts the cancellation **dates** of that cashier that
  are earlier than the operation time, divided by the number of earlier operations.

In [6]:
# STEP 4: Monetary, time and behaviour features, using the past only.
f = analytical.copy()
f['local_time'] = f['AddedAtUtc'].dt.tz_localize('UTC').dt.tz_convert('Europe/Paris')
f = f.sort_values(['AddedAtUtc', 'Id']).reset_index(drop=True)
f['amount'] = f['Amount']
f['amount_log_signed'] = np.sign(f['amount']) * np.log1p(f['amount'].abs())
f['hour'] = f['local_time'].dt.hour
f['weekday'] = f['local_time'].dt.weekday
f['month'] = f['local_time'].dt.month
f['is_saturday'] = f['weekday'].eq(5).astype('int8')

by_reg = f.groupby('CaisseId')
f['minutes_since_previous_on_register'] = by_reg['AddedAtUtc'].diff().dt.total_seconds() / 60
reg_median = by_reg['amount'].transform(lambda s: s.shift(1).expanding(min_periods=10).median())
f['gap_vs_register_median'] = f['amount'] - reg_median
counts = []
for _, g in f.groupby('CaisseId'):
    s = pd.Series(1, index=g['AddedAtUtc']).rolling('60min').count().to_numpy() - 1
    counts.append(pd.Series(s, index=g.index))
f['ops_last_60min_register'] = pd.concat(counts).sort_index()

by_user = f.groupby('AddedByUserId')
u_mean = by_user['amount'].transform(lambda s: s.shift(1).expanding(min_periods=10).mean())
u_std = by_user['amount'].transform(lambda s: s.shift(1).expanding(min_periods=10).std())
f['amount_z_vs_user'] = (f['amount'] - u_mean) / u_std.where(u_std > 0)
f['day'] = f['local_time'].dt.date
f['user_ops_today_before'] = f.groupby(['AddedByUserId', 'day']).cumcount()

past_rate = pd.Series(np.nan, index=f.index)
for _, g in f.groupby('AddedByUserId'):
    cancel_times = np.sort(g['CancelledAtUtc'].dropna().to_numpy())
    already = np.searchsorted(cancel_times, g['AddedAtUtc'].to_numpy(), side='left')
    previous_ops = np.arange(len(g))
    past_rate[g.index] = np.where(previous_ops >= 20, already / np.maximum(previous_ops, 1), np.nan)
f['user_past_cancel_rate'] = past_rate

f['register'] = f['register'].fillna('NON_RENSEIGNE')
f['cashier'] = f['cashier'].fillna('NON_RENSEIGNE')
f['is_supplier_payment'] = f['is_supplier_payment'].fillna(False).astype('int8')
display(f[['Id', 'local_time', 'register', 'cashier', 'amount', 'gap_vs_register_median', 'amount_z_vs_user',
           'ops_last_60min_register', 'user_past_cancel_rate', 'is_cancelled']].head(8))

,Id,local_time,register,cashier,amount,gap_vs_register_median,amount_z_vs_user,ops_last_60min_register,user_past_cancel_rate,is_cancelled
0,37A91C8C-B750-584B-8C54-8AA72FB131F3,2024-10-12 11:17:00+02:00,Caisse melanie,melanie,23.524,NaN,NaN,0.0,NaN,False
1,B642F892-82E1-55DA-BCDC-AE735CC42337,2024-10-12 11:35:00+02:00,Caisse melanie,melanie,14.937,NaN,NaN,1.0,NaN,False
2,443E9330-39FB-5854-8166-26DA75F96E09,2024-10-12 12:21:00+02:00,Caisse melanie,melanie,50.399,NaN,NaN,1.0,NaN,False
3,C5749451-FD9B-583D-9BAF-555FDF3EEF25,2024-10-12 12:35:00+02:00,Caisse melanie,melanie,9.558,NaN,NaN,1.0,NaN,False
4,ECF71748-D9BC-51ED-B7E7-414E12C14440,2024-10-12 12:37:00+02:00,Caisse melanie,melanie,23.667,NaN,NaN,2.0,NaN,False
5,85621E36-18FE-5A43-A75A-B186FFFF2B4B,2024-10-12 12:44:00+02:00,Caisse melanie,melanie,161.297,NaN,NaN,3.0,NaN,False
6,D5C1ED46-2C71-550B-B195-B2AF049F9DCE,2024-10-12 13:03:00+02:00,Caisse melanie,melanie,43.411,NaN,NaN,4.0,NaN,False
7,64EE479D-273E-5CAA-A196-66499021BB61,2024-10-12 13:30:00+02:00,Caisse melanie,melanie,8.796,NaN,NaN,4.0,NaN,False


## 5 — Feature Selection and Leakage Audit

### Guided code explanation — Feature selection

- Missingness, constants and the correlation heatmap are checked as in the other modules.
- `amount` and `amount_log_signed` are nearly redundant: only the log version is kept (it compresses the 56,000 € and
  −5,900 € extremes without hiding them).
- The crosstab `register × cashier` shows that each register is used by the cashier it is named after: keeping both
  would count the same information twice, so `register` is dropped and `cashier` kept.
- A feature where one value covers more than 99 % of the rows is **near-constant** (here `payment_mode`, mostly
  unknown in the source) and is dropped.
- `mean |r| with is_cancelled` is shown **for information only**: selection is not driven by the label, so that the
  unsupervised sub-case is not tuned on the answer.
- The audit table lists what must never enter the model.

In [7]:
# STEP 5: Inputs, redundancy and leakage audit.
numeric_candidates = ['amount', 'amount_log_signed', 'gap_vs_register_median', 'amount_z_vs_user', 'hour',
                      'weekday', 'month', 'is_saturday', 'minutes_since_previous_on_register',
                      'ops_last_60min_register', 'user_ops_today_before', 'user_past_cancel_rate',
                      'is_supplier_payment', 'is_walk_in_customer']
categorical_candidates = ['register', 'cashier', 'payment_mode']
leakage = ['Status', 'CancelledAtUtc', 'CancelledByUserId', 'is_cancelled']
audit_only = ['Id', 'CaisseId', 'PaymentId', 'TransactionId', 'AddedAtUtc'] + leakage

display(f[numeric_candidates + categorical_candidates].isna().mean().mul(100).round(2).to_frame('missing_%'))
corr = f[numeric_candidates].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90); ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax); ax.set_title('Numeric candidates: correlation')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()
print('Correlation with is_cancelled (information only):')
display(f[numeric_candidates].corrwith(f['is_cancelled'].astype(float)).round(3).to_frame('r'))

display(pd.crosstab(f['register'], f['cashier']))
dominant = {c: f[c].value_counts(normalize=True, dropna=False).iloc[0] for c in numeric_candidates + categorical_candidates}
redundant = {'amount', 'register'}       # register is named after its cashier: almost a 1-to-1 duplicate
constant = [c for c, share in dominant.items() if share > 0.99]   # one value covers > 99 % of rows
print('Near-constant features removed:', constant)
num_cols = [c for c in numeric_candidates if c not in redundant | set(constant)]
cat_cols = [c for c in categorical_candidates if c not in redundant | set(constant)]
selection = pd.DataFrame({'feature': numeric_candidates + categorical_candidates + audit_only})
selection['selected'] = selection['feature'].isin(num_cols + cat_cols)
selection['reason_if_dropped'] = np.select(
    [selection.feature.isin(redundant), selection.feature.isin(constant), selection.feature.isin(leakage),
     selection.feature.isin(audit_only)],
    ['redundant (log amount / cashier)', 'near-constant (> 99 % one value)', 'LEAKAGE: known only after the operation',
     'traceability only'], '')
display(selection)
selection.to_csv(OUTPUT_DIR / 'feature_selection.csv', index=False)

,missing_%
amount,0.00
amount_log_signed,0.00
gap_vs_register_median,0.49
amount_z_vs_user,0.49
hour,0.00
weekday,0.00
month,0.00
is_saturday,0.00
minutes_since_previous_on_register,0.06
ops_last_60min_register,0.00


Correlation with is_cancelled (information only):


C:\Users\firas\AppData\Local\Temp\ipykernel_13036\3343691404.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()


,r
amount,0.008
amount_log_signed,0.011
gap_vs_register_median,0.007
amount_z_vs_user,0.037
hour,-0.060
weekday,0.013
month,0.022
is_saturday,0.000
minutes_since_previous_on_register,0.030
ops_last_60min_register,-0.031


cashier,Administrateur,Celine,Hugo,Marie,kady,melanie
register,,,,,,
Caisse Celine,1,40,0,0,0,0
Caisse Hugo,0,0,1,0,0,0
Caisse Marie,0,0,0,4458,0,0
Caisse admin,23,0,0,0,0,0
Caisse kady,0,0,0,0,7,0
Caisse melanie,0,0,0,0,0,5177


Near-constant features removed: ['is_supplier_payment', 'payment_mode']


,feature,selected,reason_if_dropped
0,amount,False,redundant (log amount / cashier)
1,amount_log_signed,True,
2,gap_vs_register_median,True,
3,amount_z_vs_user,True,
4,hour,True,
5,weekday,True,
6,month,True,
7,is_saturday,True,
8,minutes_since_previous_on_register,True,
9,ops_last_60min_register,True,


## 6 — Imputation, One-Hot Encoding, Standardization and PCA

Median imputation (training period), `NON_RENSEIGNE` for missing categories, one-hot encoding, StandardScaler, then a
PCA keeping 90 % of the variance. Everything is fitted on the **earliest 80 % of operations** (chronological split).

In [8]:
# STEP 6: Chronological split and preprocessing fitted on the training period only.
ordered = f.sort_values('AddedAtUtc').reset_index(drop=True)
split = int(len(ordered) * 0.8)
train, holdout = ordered.iloc[:split].copy(), ordered.iloc[split:].copy()
print(f'Train: {len(train):,} operations up to {train.AddedAtUtc.max():%d/%m/%Y} | holdout: {len(holdout):,}')
print(f'Cancelled: train {int(train.is_cancelled.sum())} | holdout {int(holdout.is_cancelled.sum())}')

num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='NON_RENSEIGNE')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', num_pipeline, num_cols), ('cat', cat_pipeline, cat_cols)],
                                 remainder='drop', sparse_threshold=0)
X_train = preprocessor.fit_transform(train)
X_holdout = preprocessor.transform(holdout)
names = preprocessor.get_feature_names_out()
assert np.isfinite(X_train).all() and np.isfinite(X_holdout).all()
pca = PCA(n_components=0.90, svd_solver='full')
Z_train, Z_holdout = pca.fit_transform(X_train), None
Z_holdout = pca.transform(X_holdout)
print('Feature matrix:', X_train.shape, '| PCA:', X_train.shape[1], '→', Z_train.shape[1],
      '| explained variance:', round(float(pca.explained_variance_ratio_.sum()), 4))

Train: 7,765 operations up to 27/03/2025 | holdout: 1,942
Cancelled: train 93 | holdout 9


Feature matrix: (7765, 18) | PCA: 18 → 9 | explained variance: 0.9166


## 7 — Visual Evidence for the Presentation

In [9]:
# STEP 7: Amount distribution, activity by hour, discrepancies, PCA variance.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(f['amount_log_signed'], bins=50)
axes[0].set(title='Operation amount (signed log)', xlabel='sign × log(1+|amount|)')
f.groupby('hour').size().plot.bar(ax=axes[1], title='Operations per hour (Paris time)')
d = discrepancies.assign(kind=np.where(discrepancies.overnight_removal, 'opening, drawer empty', discrepancies.detected_at))
d.groupby('kind')['difference'].apply(lambda s: s.abs().median()).plot.barh(ax=axes[2], title='Median |discrepancy| by type (€)')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'cash_signals.png', dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker='o')
ax.axhline(0.90, linestyle='--', label='90 % threshold')
ax.set(xlabel='components', ylabel='cumulative explained variance', title='PCA variance retained'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()

C:\Users\firas\AppData\Local\Temp\ipykernel_13036\1071140589.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'cash_signals.png', dpi=160); plt.show()


C:\Users\firas\AppData\Local\Temp\ipykernel_13036\1071140589.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()


## 8 — Final Validation and Deliverables

In [10]:
# STEP 8: Checks and exports.
assert f['Id'].is_unique and len(f) == len(base)
assert not any(any(l in n for l in ['Status', 'Cancelled', 'is_cancelled']) for n in names)
f.drop(columns=['day']).to_csv(OUTPUT_DIR / 'cash_engineered_features.csv', index=False)
discrepancies.to_csv(OUTPUT_DIR / 'cash_discrepancies.csv', index=False)
pd.DataFrame(X_train, columns=names).to_csv(OUTPUT_DIR / 'X_train_encoded.csv', index=False)
pd.DataFrame(X_holdout, columns=names).to_csv(OUTPUT_DIR / 'X_holdout_encoded.csv', index=False)
pd.DataFrame(Z_train).to_csv(OUTPUT_DIR / 'X_train_pca.csv', index=False)
pd.DataFrame(Z_holdout).to_csv(OUTPUT_DIR / 'X_holdout_pca.csv', index=False)
summary = pd.DataFrame([
    ['raw_cash_rows', len(cash)], ['retained_operations', len(base)], ['quarantined_operations', len(quarantine)],
    ['cancelled_operations', int(f.is_cancelled.sum())], ['treasury_match_%', round(100 * f.treasury_id.notna().mean(), 2)],
    ['discrepancies', len(discrepancies)], ['overnight_removals', int(discrepancies.overnight_removal.sum())],
    ['encoded_feature_count', X_train.shape[1]], ['pca_component_count', Z_train.shape[1]],
], columns=['metric', 'value'], dtype=object)
display(summary)
summary.to_csv(OUTPUT_DIR / 'preparation_summary.csv', index=False)

,metric,value
0,raw_cash_rows,9707
1,retained_operations,9707
2,quarantined_operations,0
3,cancelled_operations,102
4,treasury_match_%,100.0
5,discrepancies,437
6,overnight_removals,190
7,encoded_feature_count,18
8,pca_component_count,9


# 9 — Optional extension A: unsupervised anomaly scoring

Isolation Forest trained on the earlier operations, applied to the later ones. `contamination = 0.02` is a **review
budget**. Without labelled anomalies we cannot measure accuracy; as an indication we compare the share of
**cancelled** operations among the alerts with the overall share (cancellations were not used by the model).

In [11]:
# STEP 9: Isolation Forest review queue.
from sklearn.ensemble import IsolationForest

iso = IsolationForest(n_estimators=300, contamination=0.02, random_state=42, n_jobs=-1).fit(X_train)
holdout['anomaly_score'] = iso.decision_function(X_holdout)
holdout['flagged_for_review'] = iso.predict(X_holdout) == -1
print(f'Alerts: {int(holdout.flagged_for_review.sum())} of {len(holdout):,} operations')
print(f'Cancelled among alerts: {100 * holdout.loc[holdout.flagged_for_review, "is_cancelled"].mean():.1f} % '
      f'vs {100 * holdout.is_cancelled.mean():.1f} % overall')
review_cols = ['Id', 'local_time', 'register', 'cashier', 'amount', 'gap_vs_register_median', 'amount_z_vs_user',
               'ops_last_60min_register', 'anomaly_score', 'is_cancelled']
queue = holdout.loc[holdout.flagged_for_review, review_cols].sort_values('anomaly_score')
queue.to_csv(OUTPUT_DIR / 'cash_anomaly_review_queue.csv', index=False)
display(queue.head(12))

Alerts: 35 of 1,942 operations
Cancelled among alerts: 5.7 % vs 0.5 % overall


,Id,local_time,register,cashier,amount,gap_vs_register_median,amount_z_vs_user,ops_last_60min_register,anomaly_score,is_cancelled
7870,6C36E9EE-1FD4-5FE2-AA26-54FD5522BF54,2025-03-29 00:00:00+01:00,Caisse Marie,Marie,1185.255,1139.4300,9.042518,0.0,-0.143188,False
9406,A100FFA9-D0E5-5287-8432-4EFCB062C03B,2025-04-26 02:40:00+02:00,Caisse Celine,Celine,189.315,82.8150,-0.156255,0.0,-0.142928,False
8967,0D969A59-5EA2-5837-A4B6-2D31B4D4AA9C,2025-04-19 00:00:00+02:00,Caisse Marie,Marie,2737.424,2691.6470,20.259367,1.0,-0.123234,False
8541,8C54FEF2-AAE0-5788-B37C-743F1174679C,2025-04-12 00:00:00+02:00,Caisse Marie,Marie,300.000,254.1750,1.710684,0.0,-0.095677,False
8919,AED89029-6417-5199-A930-B65C03BD8E3D,2025-04-18 14:00:00+02:00,Caisse Celine,Celine,30.000,-78.0000,-0.175970,0.0,-0.086814,False
8070,4553B7FA-28DF-5B5C-9934-DB01AEDB8F40,2025-04-04 01:00:00+02:00,Caisse Marie,Marie,663.039,616.8050,4.747513,0.0,-0.082865,False
8964,969116EF-82B8-56C8-8F5D-9828397F9396,2025-04-19 00:00:00+02:00,Caisse Marie,Marie,307.910,262.1260,1.724413,2.0,-0.077372,False
9062,DF70C022-F907-570A-8967-75BB3ECEFFB0,2025-04-22 10:25:00+02:00,Caisse kady,kady,-4760.900,NaN,NaN,0.0,-0.069997,False
8351,E53854BC-0234-5807-BD81-E8D54F6E5A28,2025-04-10 00:00:00+02:00,Caisse Marie,Marie,290.000,243.9985,1.769526,0.0,-0.068706,False
7873,1369C333-54A4-529F-B95F-F0DBDAFCEB5C,2025-03-29 10:25:00+01:00,Caisse Marie,Marie,413.772,367.9640,2.754239,1.0,-0.064752,False


# 10 — Optional extension B: which operations are likely to be cancelled?

The cahier allows a classification sub-case: `is_cancelled` is a real label. Cancellations are **rare** (about 1 %),
so:
- `class_weight='balanced'` gives the rare class more weight during training;
- **accuracy is useless** (predicting "never cancelled" is 99 % accurate); we report **precision, recall and average
  precision (PR-AUC)**, and compare PR-AUC with the base rate (the score of a random ranking).

A predicted cancellation is a hint to supervise the operation, not an accusation.

In [12]:
# STEP 10: Logistic regression on the same prepared matrix, chronological evaluation.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score

y_train, y_hold = train['is_cancelled'].astype(int), holdout['is_cancelled'].astype(int)
clf = LogisticRegression(class_weight='balanced', max_iter=2000).fit(X_train, y_train)
proba = clf.predict_proba(X_holdout)[:, 1]
top = np.argsort(-proba)[:max(1, int(y_hold.sum()) * 2)]           # review twice as many as the true cancellations
pred_top = np.zeros(len(proba), dtype=int); pred_top[top] = 1
report = pd.DataFrame([
    ['holdout cancellations', int(y_hold.sum())],
    ['base rate (random ranking PR-AUC)', round(float(y_hold.mean()), 4)],
    ['model PR-AUC (average precision)', round(float(average_precision_score(y_hold, proba)), 4)],
    [f'precision when reviewing top {len(top)}', round(float(precision_score(y_hold, pred_top, zero_division=0)), 3)],
    [f'recall when reviewing top {len(top)}', round(float(recall_score(y_hold, pred_top, zero_division=0)), 3)],
], columns=['metric', 'value'])
display(report)
report.to_csv(OUTPUT_DIR / 'cancellation_classifier_report.csv', index=False)
coef = pd.Series(clf.coef_[0], index=names).sort_values()
display(pd.concat([coef.head(5), coef.tail(5)]).round(3).to_frame('logistic coefficient'))

# A holdout with a handful of cancellations is unstable: add a time-series cross-validation on the training period.
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
folds = []
for k, (a, b) in enumerate(TimeSeriesSplit(n_splits=5).split(X_train), 1):
    if y_train.iloc[b].sum() == 0:
        continue
    m = clone(clf).fit(X_train[a], y_train.iloc[a])
    folds.append([k, int(y_train.iloc[b].sum()), round(float(y_train.iloc[b].mean()), 4),
                  round(float(average_precision_score(y_train.iloc[b], m.predict_proba(X_train[b])[:, 1])), 4)])
folds = pd.DataFrame(folds, columns=['fold', 'cancellations', 'base_rate', 'PR_AUC'])
display(folds)
print('Mean PR-AUC / mean base rate:', round(folds.PR_AUC.mean() / folds.base_rate.mean(), 2))

C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,metric,value
0,holdout cancellations,9.0000
1,base rate (random ranking PR-AUC),0.0046
2,model PR-AUC (average precision),0.0073
3,precision when reviewing top 18,0.0000
4,recall when reviewing top 18,0.0000


,logistic coefficient
cat__cashier_melanie,-1.492
cat__cashier_Marie,-1.351
cat__cashier_Celine,-0.773
num__hour,-0.345
num__ops_last_60min_register,-0.244
num__month,0.265
num__is_walk_in_customer,0.467
num__user_past_cancel_rate,0.923
cat__cashier_kady,1.656
cat__cashier_Administrateur,2.734


C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,fold,cancellations,base_rate,PR_AUC
0,1,23,0.0178,0.1016
1,2,6,0.0046,0.0085
2,3,16,0.0124,0.0084
3,4,11,0.0085,0.0125
4,5,11,0.0085,0.0090


Mean PR-AUC / mean base rate: 2.7


### Reading the result

PR-AUC must be compared with the **base rate**: a random ranking scores about the share of cancellations. A ratio
close to 1 means the variables available at creation time carry **little signal** about a future cancellation; a
ratio of 2 or more means the ranking is useful for supervision.

With about one hundred cancellations in the real period and a handful in the holdout, conclusions are fragile. The
honest recommendation is: keep sub-case A (review queue) for the presentation, and present sub-case B as a tested
option that needs more history (the generated months can be added with `INCLUDE_SYNTHETIC = True` for a technical
test only).

## 11 — What to say during the presentation

**Person 1:** "We profiled the five cash and treasury sources, repaired the malformed event log, converted UTC dates
and amounts with explicit formats and kept negative and extreme amounts. We built monetary and time features in Paris
time, always compared with the past of the same register or cashier."

**Person 2:** "We verified the real key chain between operations, treasury records and payments, because the first
version's joins matched nothing. We added register and cashier names, separated overnight cash removals from real
discrepancies, excluded every variable known only after a cancellation, then imputed, encoded, scaled and tested a
PCA."

**Both:** "Sub-case A gives a review queue; sub-case B ranks operations by cancellation risk. Both are aids for
supervision, not proof of fraud."

### Limitations to disclose
- Payment mode is missing for most payments in the source export.
- 102 cancellations in the real period: the classifier is evaluated on a few dozen positives.
- The alert budget (2 %) and the review size are business choices.
- No confirmed fraud label exists: precision of sub-case A cannot be measured without human review.